In [5]:
"""
Taller 1 - Teoria de Aprendizaje de Maquina - 2026-2
Ejecucion: python codigo_taller.py
Dependencias: numpy, matplotlib, scikit-learn
"""
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score
)
from sklearn.inspection import DecisionBoundaryDisplay

# ============================================================
# Punto 1.1: frontera de decision del clasificador lineal
# ============================================================
# Definición de los pesos (w) y el sesgo (b) del clasificador lineal
w = np.array([1, 2])
b = -3

# Puntos tridimensionales para evaluar la frontera
puntos = np.array([[0, 0], [1, 0], [2, 1], [3, 1]])

# Evaluamos cada punto aplicando la ecuación w^T * x + b
for i, x in enumerate(puntos, start=1):
    score = w @ x + b
    prediccion = 1 if score > 0 else -1
    print(f"x{i}={x}: w^T x + b = {score}; clase = {prediccion}")

# La frontera cumple x1 + 2*x2 - 3 = 0, es decir x2=(3-x1)/2.
x1_recta = np.linspace(-0.5, 3.5, 200)
x2_recta = (3 - x1_recta) / 2

# Graficamos la frontera de decisión y los puntos evaluados
plt.figure(figsize=(7, 5))
plt.plot(x1_recta, x2_recta, label="Frontera: x1 + 2x2 - 3 = 0")
scores = puntos @ w + b
colores = np.where(scores > 0, "tab:blue", "tab:orange")
plt.scatter(puntos[:, 0], puntos[:, 1], c=colores, s=70, edgecolor="black")
for i, (x, score) in enumerate(zip(puntos, scores), start=1):
    plt.annotate(f"x{i} ({1 if score > 0 else -1})", x + np.array([0.05, 0.05]))
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Frontera de decision del perceptron")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig("frontera_perceptron.png", dpi=180)
plt.close()

# ============================================================
# Punto 1.3: actualizacion numerica del perceptron
# ============================================================
# Pesos y sesgo iniciales para el cálculo de la actualización
w = np.array([0.0, 0.0])
b = 0.0
eta = 0.5
x = np.array([2.0, 1.0])
y_real = 1

# Calculamos la predicción inicial y aplicamos la regla de actualización del perceptrón si falla
score = w @ x + b
y_pred = 1 if score > 0 else -1
print("\nPunto 1.3")
print("Prediccion inicial:", y_pred)
if y_pred != y_real:
    # Regla de actualización: w_nuevo = w + eta * y_real * x
    w = w + eta * y_real * x
    b = b + eta * y_real
    margen = y_real * (w @ x + b)
    print("Nuevos pesos:", w)
    print("Nuevo sesgo:", b)
    print("Nuevo margen:", margen)
else:
    print("No se requiere actualizacion.")

# ============================================================
# Punto 2.1: distancias manuales
# ============================================================
# Punto de consulta (query q) y observaciones para calcular distancias KNN de forma manual
q = np.array([1.0, 1.0])
observaciones = np.array([
    [0.0, 0.0], [1.0, 0.0], [3.0, 1.0],
    [0.0, 3.0], [1.1, 1.1]
])
clases = np.array(["A", "A", "B", "B", "B"])

# Distancia euclídea manual a cada observación
distancias = np.linalg.norm(observaciones - q, axis=1)
orden = np.argsort(distancias)
print("\nPunto 2.1: distancias ordenadas")
for indice in orden:
    print(f"x{indice+1}: distancia={distancias[indice]:.4f}, clase={clases[indice]}")

# Predicción usando k vecinos más cercanos mediante votación mayoritaria
for k in (1, 3, 5):
    vecinos = clases[orden[:k]]
    conteo_A = np.sum(vecinos == "A")
    conteo_B = np.sum(vecinos == "B")
    clase_predicha = "A" if conteo_A > conteo_B else "B"
    print(f"k={k}: vecinos={list(vecinos)}, votos A={conteo_A}, "
          f"votos B={conteo_B}, prediccion={clase_predicha}")

# ============================================================
# Punto 2.2: experimento make_moons y regiones de decision
# ============================================================
# Generamos y dividimos el conjunto de datos sintético "make_moons"
X, y = make_moons(n_samples=300, noise=0.20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

modelos = {}
print("\nPunto 2.2: exactitudes")
# Entrenamos clasificadores KNN para diferentes valores de k y calculamos el accuracy
for k in (1, 5, 15):
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_train, y_train)
    modelos[k] = modelo
    acc_train = accuracy_score(y_train, modelo.predict(X_train))
    acc_test = accuracy_score(y_test, modelo.predict(X_test))
    print(f"k={k:2d}: entrenamiento={acc_train:.4f}, prueba={acc_test:.4f}")

# Graficamos la distribución original de los datos de entrenamiento y prueba
plt.figure(figsize=(7, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm",
            s=25, edgecolor="black", linewidth=0.35, label="Entrenamiento")
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm",
            s=45, marker="x", linewidth=1.2, label="Prueba")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Datos make_moons: entrenamiento y prueba")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("datos_moons.png", dpi=180)
plt.close()

# Visualizamos las regiones y fronteras de decisión para k=1 y k=15 usando DecisionBoundaryDisplay
for k in (1, 15):
    plt.figure(figsize=(7, 5))
    ax = plt.gca()
    DecisionBoundaryDisplay.from_estimator(
        modelos[k], X, response_method="predict",
        cmap="coolwarm", alpha=0.28, ax=ax, grid_resolution=250
    )
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm",
               s=22, edgecolor="black", linewidth=0.3, label="Entrenamiento")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(f"Regiones de decision para k={k}")
    ax.legend()
    ax.grid(alpha=0.15)
    plt.tight_layout()
    plt.savefig(f"frontera_k{k}.png", dpi=180)
    plt.close()

# ============================================================
# Punto 3: matriz de confusion y metricas
# ============================================================
# Vectores de etiquetas reales e inferidas por un clasificador ficticio para evaluación de métricas
y_real = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
y_pred = np.array([0, 1, 0, 0, 1, 0, 1, 1, 0, 1])

# Cálculo de la matriz de confusión y obtención de verdaderos/falsos positivos/negativos
matriz = confusion_matrix(y_real, y_pred, labels=[0, 1])
tn, fp, fn, tp = matriz.ravel()
print("\nPunto 3: matriz de confusion")
print(matriz)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
# Cálculo de métricas clásicas de evaluación: Accuracy, Precision, Recall y F1-Score
print("Accuracy:", accuracy_score(y_real, y_pred))
print("Precision:", precision_score(y_real, y_pred, zero_division=0))
print("Recall:", recall_score(y_real, y_pred, zero_division=0))
print("F1:", f1_score(y_real, y_pred, zero_division=0))


x1=[0 0]: w^T x + b = -3; clase = -1
x2=[1 0]: w^T x + b = -2; clase = -1
x3=[2 1]: w^T x + b = 1; clase = 1
x4=[3 1]: w^T x + b = 2; clase = 1

Punto 1.3
Prediccion inicial: -1
Nuevos pesos: [1.  0.5]
Nuevo sesgo: 0.5
Nuevo margen: 3.0

Punto 2.1: distancias ordenadas
x5: distancia=0.1414, clase=B
x2: distancia=1.0000, clase=A
x1: distancia=1.4142, clase=A
x3: distancia=2.0000, clase=B
x4: distancia=2.2361, clase=B
k=1: vecinos=[np.str_('B')], votos A=0, votos B=1, prediccion=B
k=3: vecinos=[np.str_('B'), np.str_('A'), np.str_('A')], votos A=2, votos B=1, prediccion=A
k=5: vecinos=[np.str_('B'), np.str_('A'), np.str_('A'), np.str_('B'), np.str_('B')], votos A=2, votos B=3, prediccion=B

Punto 2.2: exactitudes
k= 1: entrenamiento=1.0000, prueba=0.9333
k= 5: entrenamiento=0.9571, prueba=0.9556
k=15: entrenamiento=0.9571, prueba=0.9333

Punto 3: matriz de confusion
[[3 1]
 [2 4]]
TN=3, FP=1, FN=2, TP=4
Accuracy: 0.7
Precision: 0.8
Recall: 0.6666666666666666
F1: 0.7272727272727273
